In [ ]:
import torch
import torchvision.utils as vutils
from Network import VAE_41

In [ ]:
#from torch.utils.data import random_split
#from torch.utils.data import DataLoader
import numpy as np
batch_size = 64
file_dir = r"..\v41_data\train_data.npy"
data_np = np.load(file_dir)


In [ ]:
def standarlized(input_images):    
    min_val = np.min(input_images)
    max_val = np.max(input_images)
    # 归一化操作，将数据范围从 [min_val, max_val] 映射到 [0, 1]
    normalized_images = (input_images - min_val) / (max_val - min_val)
    return normalized_images

data_clean = standarlized(data_np)

In [ ]:
# 加载模型权重
model = VAE_41()   # 这里应该是你的模型定义
model.load_state_dict(torch.load('./weight/size41_0119test.pth'))  # 加载预训练的权重

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_input = torch.from_numpy(data_clean).float().unsqueeze(1)
test_dataset = data_input.to(device)
model.to(device)
model.eval()
latent_dim = 16 

sample_idx = 500 # 选择一个测试样本
x = test_dataset[sample_idx]
x = x.unsqueeze(0).to(device)
# 1. 获取基准分布
with torch.no_grad():
    mu, logvar = model.encode(x)
    sigma = torch.exp(0.5 * logvar)
    steps = torch.tensor([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0], device=mu.device)  

# 2. 遍历前 5 个维度 (假设总维度较多)
all_images = []
for d in range(16):
    # 针对每一个维度，生成从 -3 到 3 的 8 张图
    temp_mu = mu.repeat(7, 1)
    #temp_mu[:, d] = torch.linspace(-3, 3, 7)
    temp_mu[:, d] = mu[:, d] + steps * sigma[:, d]
    

    with torch.no_grad():
        #print(temp_mu.shape)
        decoded = model.decode(temp_mu)
        all_images.append(decoded)
        
    

#print(len(all_images))
# 3. 拼接成网格并保存/显示
#grid = vutils.make_grid(torch.cat(all_images), nrow=8)
# 使用 matplotlib 或 save_image 查看 grid

In [ ]:

import pandas as pd

min_values_matrix = []

for dim_idx in range(len(all_images)):
    # 拿到当前维度的 7 张图 [7, 1, 41, 41]
    dim_images = all_images[dim_idx]
    
    # 对这 7 张图分别求最小值
    # view(7, -1) 将每张图展平，然后在第 1 维（像素维）求最小值
    mins, _ = torch.min(dim_images.view(7, -1), dim=1)
    
    # 转为列表存入
    min_values_matrix.append(mins.tolist())

# 定义列名（扰动程度）
column_names = ["-3σ", "-2σ", "-1σ", "Original", "+1σ", "+2σ", "+3σ"]
# 定义行名（隐变量维度）
row_names = [f"Latent_Dim_{i}" for i in range(16)]

# 生成 DataFrame
df = pd.DataFrame(min_values_matrix, columns=column_names, index=row_names)

# 打印表格
#print("--- 16x7 最小值分布表 ---")
#print(df)

In [ ]:
import matplotlib.pyplot as plt

# 设置画布大小
plt.figure(figsize=(12, 7))

# 遍历每一行数据（每一个维度）
for i in range(len(df)):
    # df.iloc[i] 对应一个维度在 7 个 sigma 下的最小值
    plt.plot(column_names, df.iloc[i], marker='o', label=df.index[i] if i < 10 else "")

# 优化图表
plt.title("Min Value Trend across Latent Dimensions (-3σ to +3σ)", fontsize=14)
plt.xlabel("Perturbation Range", fontsize=12)
plt.ylabel("Matrix Min Value", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)

# 如果维度太多，图例会很乱，这里只显示前10个
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), title="Dimensions")
plt.tight_layout()

plt.show()

In [ ]:
original_1 = test_dataset[501].min().item()
original_2 = test_dataset[502].min().item()

plt.figure(figsize=(12, 8))

# 2. 绘制 16 个维度的扰动曲线
for i in range(len(df)):
    plt.plot(column_names, df.iloc[i], marker='o', alpha=0.7, label=f"Dim {i}" if i < 8 else "")

# 3. 绘制原始样本的基准线 (红色虚线)
plt.axhline(y=original_1, color='r', linestyle='--', linewidth=2, label='load')
plt.axhline(y=original_2, color='g', linestyle='--', linewidth=2, label='next_load')

# 4. 图表美化
plt.title("Latent Traversal vs. Original Baseline", fontsize=14)
plt.xlabel("Perturbation (Sigma Steps)", fontsize=12)
plt.ylabel("Matrix Minimum Value", fontsize=12)
plt.grid(True, which='both', linestyle=':', alpha=0.5)

# 添加标注，说明红线是什么
#plt.text(0, , f' Base: {original_min:.4f}', color='red', va='bottom', fontweight='bold')

plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
recon_x = model(x)[0]
print(recon_x.shape)

In [ ]:
# 确保 grid 在 CPU 上，并且转换维度从 [C, H, W] 到 [H, W, C]
grid_np = grid.permute(1, 2, 0).cpu().numpy()

# 如果是灰度图，grid_np 虽然有 3 个通道，但数值一样
# 直接指定 gray 映射显示最稳妥
plt.imshow(grid_np[:, :, 0], cmap='viridis') 
plt.axis('off')
plt.title("Latent Space Traversal")
plt.show()

In [ ]:
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# 1. 数据处理
# 将列表中的 5 个 [8, 1, 41, 41] 拼接成一个 [40, 1, 41, 41] 的大张量
all_tensors = torch.cat(all_images, dim=0) 

# 2. 生成 Grid
# nrow=8 表示每行显示一个维度的 8 次变化
# normalize=True 会自动处理模糊或区间不一致的问题
grid = vutils.make_grid(all_tensors, nrow=8, normalize=True, pad_value=1)

# 3. 转换为 Numpy 格式以便 Matplotlib 显示
# grid 形状从 [3, H, W] 转为 [H, W, 3]
grid_np = grid.permute(1, 2, 0).cpu().numpy()

# 4. 绘图与标注
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(grid_np[:, :, 0], cmap='viridis') # 取单通道显示灰度

# 计算标注位置
rows = 5
cols = 8
h, w = grid_np.shape[:2]

# --- 标注维度 (Y轴) ---
# 假设你遍历的是维度 0, 1, 2, 3, 4
dim_indices = [0, 1, 2, 3, 4] 
row_centers = [int((i + 0.5) * (h / rows)) for i in range(rows)]
ax.set_yticks(row_centers)
ax.set_yticklabels([f"Dimension {d}" for d in dim_indices], fontsize=12, fontweight='bold')

# --- 标注变化值 (X轴) ---
# 假设你的遍历区间是 -3 到 3
traversal_values = torch.linspace(-3, 3, steps=cols)
col_centers = [int((j + 0.5) * (w / cols)) for j in range(cols)]
ax.set_xticks(col_centers)
ax.set_xticklabels([f"{v.item():.1f}" for v in traversal_values], fontsize=10)

# 装饰
ax.set_title("VAE Latent Space Traversal (41x41)", fontsize=16, pad=20)
ax.set_xlabel("Latent Value ($\mu + \sigma \cdot \epsilon$)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torchvision.utils as vutils
import matplotlib.pyplot as plt

# 假设 latent_dim = 16
# 假设你已经准备好了 all_images_16 (List, 包含16个 [8, 1, 41, 41] 的 Tensor)

# 1. 拼接 16 个维度的结果
traversal_tensors = torch.cat(all_images, dim=0) # 形状 [128, 1, 41, 41]
grid_traversal = vutils.make_grid(traversal_tensors, nrow=7, normalize=True, pad_value=1)
grid_traversal_np = grid_traversal.permute(1, 2, 0).cpu().numpy()[:, :, 0]

# 2. 创建画布
# 因为有16行，高度要设大一点，比如 20
fig = plt.figure(figsize=(20, 24), constrained_layout=True)
gs = fig.add_gridspec(16, 6) # 16行，6列布局

# --- 左侧：参考区 (占据前两行，第一列) ---
ax_orig = fig.add_subplot(gs[0:2, 0]) 
ax_orig.imshow(x.squeeze().cpu().numpy(), cmap='viridis')
ax_orig.set_title("Original", fontsize=16)
ax_orig.axis('off')

ax_recon = fig.add_subplot(gs[2:4, 0])
ax_recon.imshow(recon_x.detach().squeeze().cpu().numpy(), cmap='viridis')
ax_recon.set_title("Reconstruction", fontsize=16)
ax_recon.axis('off')

# --- 右侧：16维全遍历区 (占据所有行，剩余5列) ---
ax_trav = fig.add_subplot(gs[:, 1:]) 
ax_trav.imshow(grid_traversal_np, cmap='viridis')

# 标注 16 个维度
rows, cols = 16, 7
h, w = grid_traversal_np.shape
row_centers = [int((i + 0.5) * (h / rows)) for i in range(rows)]
ax_trav.set_yticks(row_centers)
ax_trav.set_yticklabels([f"Dim {i}" for i in range(rows)], fontsize=12)

# 标注横轴数值
col_centers = [int((j + 0.5) * (w / cols)) for j in range(cols)]
ax_trav.set_xticks(col_centers)
ax_trav.set_xticklabels([f"{v:.1f}" for v in torch.linspace(-3, 3, 7)], fontsize=10)

ax_trav.set_title("16-Dimension Latent Space Traversal", fontsize=20, pad=20)
plt.show()

In [ ]:
# 打印一下你那个样本的 sigma (std)
std = torch.exp(0.5 * logvar)
print(f"Sigma 的平均值: {std.mean().item()}")